In [52]:
import pandas as pd
import glob
import random
import folium
import numpy as np
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

In [53]:
csv_files = glob.glob("../data/*.csv")

df = pd.concat((pd.read_csv(csv_file) for csv_file in csv_files), ignore_index=True).dropna()

In [54]:
df = df.sort_values(
    by=["icao24", "callsign", "time"]
).reset_index(drop=True)

group_cols = ["icao24", "callsign"]

indices_to_remove = (
    df.groupby(group_cols)
      .apply(lambda x: x.index[-1] if len(x) % 2 else None)
      .dropna()
      .astype(int)
)

df = df.drop(indices_to_remove).reset_index(drop=True)

In [55]:
display(df.head())
display(df.tail())
df.shape

,time,icao24,lat,lon,velocity,heading,vertrate,callsign,onground,alert,spi,squawk,baroaltitude,geoaltitude,lastposupdate,lastcontact
0,1497271880,000001,49.004517,17.505094,16.749083,349.380345,1.95072,CLAW41,False,False,False,7070.0,1402.08,1402.08,1.497272e+09,1.497272e+09
1,1497271890,000001,49.004517,17.505094,23.201367,3.814075,0.97536,CLAW41,False,False,False,7070.0,1310.64,1303.02,1.497272e+09,1.497272e+09
2,1497271900,000001,49.004517,17.505094,23.201367,3.814075,0.97536,CLAW41,False,False,False,7070.0,1280.16,1303.02,1.497272e+09,1.497272e+09
3,1497271910,000001,49.004517,17.505094,23.201367,3.814075,0.97536,CLAW41,False,False,False,2030.0,1249.68,1303.02,1.497272e+09,1.497272e+09
4,1497271920,000001,49.004517,17.505094,23.201367,3.814075,0.97536,CLAW41,False,False,False,7070.0,1219.20,1303.02,1.497272e+09,1.497272e+09


,time,icao24,lat,lon,velocity,heading,vertrate,callsign,onground,alert,spi,squawk,baroaltitude,geoaltitude,lastposupdate,lastcontact
48726693,1497240930,ecce1b,34.045074,-117.463522,69.510884,298.272036,-3.57632,N924MB,False,False,False,7673.0,487.68,800.1,1.497241e+09,1.497241e+09
48726694,1497240940,ecce1b,34.045074,-117.463522,69.510884,298.272036,-3.57632,N924MB,False,False,False,7673.0,487.68,800.1,1.497241e+09,1.497241e+09
48726695,1497240950,ecce1b,34.045074,-117.463522,69.510884,298.272036,-3.57632,N924MB,False,False,False,7673.0,487.68,800.1,1.497241e+09,1.497241e+09
48726696,1497240960,ecce1b,34.045074,-117.463522,69.510884,298.272036,-3.57632,N924MB,False,False,False,7673.0,487.68,800.1,1.497241e+09,1.497241e+09
48726697,1497240970,ecce1b,34.045074,-117.463522,69.510884,298.272036,-3.57632,N924MB,False,False,False,7673.0,487.68,800.1,1.497241e+09,1.497241e+09


(48726698, 16)

In [56]:
trajectories = {
    key: group[["lat", "lon"]].to_dict('records')
    for key, group in df.groupby(['icao24', 'callsign'])
}

In [57]:
print(next(iter(trajectories.values())))
display(len(trajectories))

[{'lat': 49.0045166016, 'lon': 17.505094088}, {'lat': 49.0045166016, 'lon': 17.505094088}, {'lat': 49.0045166016, 'lon': 17.505094088}, {'lat': 49.0045166016, 'lon': 17.505094088}, {'lat': 49.0045166016, 'lon': 17.505094088}, {'lat': 49.0126190186, 'lon': 17.5081223708}, {'lat': 49.0210891982, 'lon': 17.5137248792}, {'lat': 49.0221862793, 'lon': 17.5143902118}, {'lat': 49.0221862793, 'lon': 17.5143902118}, {'lat': 49.0221862793, 'lon': 17.5143902118}, {'lat': 49.0221862793, 'lon': 17.5143902118}, {'lat': 49.0221862793, 'lon': 17.5143902118}, {'lat': 49.0221862793, 'lon': 17.5143902118}, {'lat': 49.0221862793, 'lon': 17.5143902118}]


107311

In [58]:
selected = random.sample(
    list(trajectories.items()),
    min(5, len(trajectories))
)

first_trajectory = selected[0][1]

m = folium.Map(
    location=[
        first_trajectory[0]['lat'],
        first_trajectory[0]['lon']
    ],
    zoom_start=6
)

for (icao24, callsign), trajectory in selected:

    points = [
        [point['lat'], point['lon']]
        for point in trajectory
        if point['lat'] is not None and point['lon'] is not None
    ]

    folium.PolyLine(
        points,
        tooltip=f"{icao24} / {callsign}",
        weight=3
    ).add_to(m)

m

In [59]:
split_index = int(len(df) * 0.995)

train_df = df.iloc[:split_index]
train_x = []
train_y = []
test_df = df.iloc[split_index:]
test_x = []
test_y = []

def make_xy(frame):
    frame = frame.sort_values(["icao24", "callsign", "time"])
    g = frame.groupby(["icao24", "callsign"])
    d = frame.assign(
        prev_lat=g["lat"].shift(1), prev_lon=g["lon"].shift(1),
        next_lat=g["lat"].shift(-1), next_lon=g["lon"].shift(-1),
        dt=g["time"].diff(),
    ).dropna(subset=["prev_lat", "prev_lon", "next_lat", "next_lon", "dt"])
    d = d[d["dt"] > 0]

    X = np.column_stack([
        d.lat, d.lon,
        (d.lat - d.prev_lat) / d.dt,   # lat speed (deg/s)
        (d.lon - d.prev_lon) / d.dt,   # lon speed (deg/s)
    ])
    y = np.column_stack([d.next_lat - d.lat, d.next_lon - d.lon])  # delta, not absolute
    return X, y

train_x, train_y = make_xy(train_df)
test_x, test_y = make_xy(test_df)

scaler = StandardScaler()
train_x = scaler.fit_transform(train_x)
test_x = scaler.transform(test_x)

In [67]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

# model = Ridge()                                                     # fast, try this first
model = RandomForestRegressor(n_estimators=100, n_jobs=-1)        # handles multi-output natively
# model = MultiOutputRegressor(HistGradientBoostingRegressor())     # usually the strongest
# model = KNeighborsRegressor(n_neighbors=5)                        # your original, for comparison

# With ~48M rows, tree models are slow, so train on a random subsample:
idx = np.random.default_rng(42).choice(len(train_x), size=min(1_000_000, len(train_x)), replace=False)
model.fit(train_x[idx], train_y[idx])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [68]:
pred = model.predict(test_x)
print("Model MSE (delta):   ", mean_squared_error(test_y, pred))

# Baseline: next delta = previous delta (the 10 s step is the training sampling rate)
raw = scaler.inverse_transform(test_x)
baseline = raw[:, 2:4] * 10
print("Baseline MSE (delta):", mean_squared_error(test_y, baseline))

Model MSE (delta):    0.3063654237463087
Baseline MSE (delta): 0.2710378647117855


In [69]:
# Get all unique flights
flights = test_df[["icao24", "callsign"]].drop_duplicates()

# Pick one random flight
flight = flights.sample(n=1).iloc[0]

icao24 = flight["icao24"]
callsign = flight["callsign"]

print(f"Selected flight: {icao24} / {callsign}")

# Get only this flight
single_flight = test_df[
    (test_df["icao24"] == icao24) &
    (test_df["callsign"] == callsign)
].sort_values("time")

# Get lat, lon, altitude
single_data = single_flight[
    ["lat", "lon"]
].to_numpy()

# Create input/output pairs
single_x = []
single_y = []

for i in range(len(single_data) - 1):
    single_x.append(single_data[i])
    single_y.append(single_data[i + 1])

single_x = np.array(single_x)
single_y = np.array(single_y)

# Scale X using the SAME scaler used for training
single_x_scaled = scaler.transform(single_x)

# Predict
single_pred = model.predict(single_x_scaled)

# Plot
plt.figure(figsize=(10, 8))

# Actual trajectory
plt.plot(
    single_y[:, 1],       # longitude
    single_y[:, 0],       # latitude
    label="Actual"
)

# Predicted trajectory
plt.plot(
    single_pred[:, 1],    # longitude
    single_pred[:, 0],    # latitude
    linestyle=":",
    label="Predicted"
)

plt.title(f"KNN Flight Trajectory - {callsign}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.legend()
plt.grid(True)
plt.axis("equal")

plt.show()

Selected flight: e48f9e / PTZNF   


ValueError: X has 2 features, but StandardScaler is expecting 4 features as input.

In [70]:
# --- Predict the next point for trajectory_rows.csv ---

rows_df = pd.read_csv("trajectory_rows.csv")

# The "trajectory" column is a JSON string: a list of
# {"lat", "lon", "altitude", "timestamp", "pred"} dicts.
rows_df["trajectory"] = rows_df["trajectory"].apply(json.loads)

rows_df[["id", "icao", "callsign"]].assign(n_points=rows_df["trajectory"].apply(len))

,id,icao,callsign,n_points
0,358,E48043,TAM3935_,5
1,359,E48989,GLO1028_,25
2,360,E48B65,TAM3042_,10
3,361,E48C30,GLO1529_,4
4,362,E48E78,GLO1115_,5
5,363,E490B1,AZU6068_,11
6,364,E4910C,AZU6068_,13
7,365,E4952F,GLO1504_,16
8,366,E49764,TAM3286_,26
9,367,E49BFD,TAM3506_,3


In [73]:
N_STEPS = 3
STEP_SECONDS = 10   # the model was trained on ~10 s steps, so each predicted step is ~10 s

results = []

for _, row in rows_df.iterrows():
    real_points = sorted(
        (p for p in row["trajectory"] if not p.get("pred", False)),
        key=lambda p: p["timestamp"],
    )
    if not real_points:
        continue

    last = real_points[-1]
    prev = next(
        (p for p in reversed(real_points) if p["timestamp"] < last["timestamp"]),
        None,
    )
    if prev is None:
        print(f"Skipping {row['callsign']}: no earlier point with a different timestamp")
        continue

    dt = last["timestamp"] - prev["timestamp"]

    # Starting state: last real position + speed measured from real data
    cur_lat, cur_lon = last["lat"], last["lon"]
    v_lat = (last["lat"] - prev["lat"]) / dt
    v_lon = (last["lon"] - prev["lon"]) / dt

    for step in range(1, N_STEPS + 1):
        x = np.array([[cur_lat, cur_lon, v_lat, v_lon]])
        d_lat, d_lon = model.predict(scaler.transform(x))[0]

        cur_lat += d_lat
        cur_lon += d_lon
        # Feed the predicted step back in as the new "previous speed"
        v_lat, v_lon = d_lat / STEP_SECONDS, d_lon / STEP_SECONDS

        results.append({
            "id": row["id"],
            "icao": row["icao"],
            "callsign": row["callsign"],
            "last_lat": last["lat"],
            "last_lon": last["lon"],
            "last_timestamp": last["timestamp"],
            "step": step,
            "pred_timestamp": last["timestamp"] + step * STEP_SECONDS,
            "pred_lat": cur_lat,
            "pred_lon": cur_lon,
        })

pred_df = pd.DataFrame(results)
pred_df

,id,icao,callsign,last_lat,last_lon,last_timestamp,step,pred_timestamp,pred_lat,pred_lon
0,358,E48043,TAM3935_,-23.6793,-46.6199,1777318081,1,1777318091,-23.679411,-46.619535
1,358,E48043,TAM3935_,-23.6793,-46.6199,1777318081,2,1777318101,-23.680303,-46.616372
2,358,E48043,TAM3935_,-23.6793,-46.6199,1777318081,3,1777318111,-23.685656,-46.604930
3,359,E48989,GLO1028_,-23.6742,-46.3674,1777318077,1,1777318087,-23.671313,-46.345478
4,359,E48989,GLO1028_,-23.6742,-46.3674,1777318077,2,1777318097,-23.668639,-46.324852
5,359,E48989,GLO1028_,-23.6742,-46.3674,1777318077,3,1777318107,-23.663817,-46.295456
6,360,E48B65,TAM3042_,-23.5983,-46.4923,1777316659,1,1777316669,-23.596598,-46.486092
7,360,E48B65,TAM3042_,-23.5983,-46.4923,1777316659,2,1777316679,-23.595872,-46.482763
8,360,E48B65,TAM3042_,-23.5983,-46.4923,1777316659,3,1777316689,-23.593506,-46.474517
9,361,E48C30,GLO1529_,-23.6946,-46.6095,1777316707,1,1777316717,-23.688512,-46.613494


In [74]:
center = [pred_df["last_lat"].mean(), pred_df["last_lon"].mean()]
m2 = folium.Map(location=center, zoom_start=10)

# Real trajectories
for _, row in rows_df.iterrows():
    pts = [
        [p["lat"], p["lon"]]
        for p in sorted(row["trajectory"], key=lambda p: p["timestamp"])
        if not p.get("pred", False)
    ]
    folium.PolyLine(pts, tooltip=f"{row['icao']} / {row['callsign']}", weight=3).add_to(m2)

# Predicted paths: last real point -> step 1 -> step 2 -> step 3
for (icao, callsign), g in pred_df.groupby(["icao", "callsign"]):
    g = g.sort_values("step")
    path = [[g.iloc[0]["last_lat"], g.iloc[0]["last_lon"]]] + g[["pred_lat", "pred_lon"]].values.tolist()
    folium.PolyLine(path, color="red", dash_array="5", weight=2).add_to(m2)

    folium.CircleMarker(path[0], radius=4, color="green", fill=True,
                        tooltip=f"Last real point - {callsign}").add_to(m2)
    for _, r in g.iterrows():
        folium.CircleMarker([r["pred_lat"], r["pred_lon"]], radius=4, color="red", fill=True,
                            tooltip=f"{callsign} - predicted step {r['step']}").add_to(m2)

m2